# Phase 01.02 — Zero-shot baseline (F1)

Runs single-frame zero-shot evaluation against the frozen validation set. Dynamic tiling is enabled, the visual token count comes from `model.num_image_token`, inference uses Vintern's native `Hermes-2` chat path, and FlashAttention remains off.


In [1]:
import os, sys
from pathlib import Path

PROJECT_ROOT = Path("/workspace/RoadBuddy")
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

os.environ.setdefault("CC", "/usr/bin/gcc")
os.environ.setdefault("CXX", "/usr/bin/g++")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from roadbuddy_common import *

os.chdir(PROJECT_ROOT)
seed_everything(SEED)
ensure_dirs()
print("Project root:", PROJECT_ROOT)
print("Model revision:", MODEL_REVISION)


Project root: /workspace/RoadBuddy
Model revision: b98f263eab246eb5269ade64edbdca8a887dc44d


In [2]:
VALIDATION_CSV = PATHS.phase1_split / "validation.csv"
DEBUG_LIMIT = 20  # Keep 20 for the first run; set None for the full baseline.

assert VALIDATION_CSV.is_file(), "Run Phase01_01_Dataset_Contract_Split_Freeze.ipynb first"
val_df = pd.read_csv(VALIDATION_CSV)
frozen_ids = json.loads((PATHS.phase1_split / "validation_sample_ids.json").read_text(encoding="utf-8"))
assert sorted(val_df.sample_id.astype(str)) == frozen_ids
print("Evaluation rows:", min(len(val_df), DEBUG_LIMIT) if DEBUG_LIMIT else len(val_df))


Evaluation rows: 20


## Load the pinned model


In [3]:
model, tokenizer = load_model_and_tokenizer(training=False)
model.img_context_token_id = tokenizer.convert_tokens_to_ids(IMG_CONTEXT_TOKEN)
assert int(model.num_image_token) > 0
assert model.template == "Hermes-2"
print("num_image_token:", model.num_image_token)


/root/venvs/roadbuddy-rtx3090-py310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/root/venvs/roadbuddy-rtx3090-py310/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


FlashAttention2 is not installed.


num_image_token: 256


## Run and checkpoint predictions

`native_chat` explicitly validates one `image_flag` per visual tile; the pinned Vintern native chat method consumes the equivalent tile flags internally.


In [4]:
predictions, metrics = evaluate_rows(model, tokenizer, val_df, limit=DEBUG_LIMIT)
run_name = "debug20" if DEBUG_LIMIT else "full"
out_dir = PATHS.zero_shot / run_name
out_dir.mkdir(parents=True, exist_ok=True)
predictions.to_csv(out_dir / "predictions.csv", index=False)
save_json(out_dir / "metrics.json", metrics)
save_json(out_dir / "config.json", {
    "model_id": MODEL_ID, "revision": MODEL_REVISION, "seed": SEED,
    "frames": 1, "dynamic_tiles": MAX_DYNAMIC_TILES, "thumbnail": USE_THUMBNAIL,
    "template": model.template, "num_image_token": int(model.num_image_token),
    "flash_attention": False, "limit": DEBUG_LIMIT,
})
display(metrics)
display(predictions.head())
print("Saved:", out_dir)


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


...

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


{'rows': 20,
 'accuracy': 0.65,
 'macro_f1': 0.5984848484848485,
 'parse_rate': 1.0}

,sample_id,group_id,question_type,answer,prediction,raw_response,correct,num_tiles
0,train_0006,464458b4_386_clip_006_0039_0048_Y,unknown,B,B,B,True,7
1,train_0040,81392443_006_clip_005_0028_0035_Y,unknown,B,B,B,True,7
2,train_0041,81392443_006_clip_005_0028_0035_Y,unknown,B,B,B,True,7
3,train_0042,a5cb63a2_035_clip_001_0000_0007_Y,unknown,D,B,B,False,7
4,train_0043,a5cb63a2_035_clip_001_0000_0007_Y,unknown,B,B,B,True,7


Saved: /workspace/RoadBuddy/outputs/phase01/zero_shot_f1/debug20


## Gate

Inspect parse rate, sample predictions, tile counts, and GPU memory. Set `DEBUG_LIMIT=None` and re-run from the configuration cell to create the full baseline before comparison.


## Nhận xét sau lần chạy Phase 01.02

**Phạm vi:** debug run trên 20 rows đầu của frozen validation; đây chưa phải đánh giá đầy đủ 298 rows.

### Kết quả

| Chỉ số | Giá trị |
|---|---:|
| Accuracy | 0.6500 |
| Macro-F1 | 0.5985 |
| Parse rate | 1.0000 |
| Đúng / tổng | 13 / 20 |

Phân bố nhãn của subset là A=3, B=11, C=2, D=4; subset nghiêng mạnh về B. Mỗi mẫu dùng 7 visual tiles, gồm tối đa 6 dynamic tiles cộng thumbnail.

### Theo lớp

- A: F1=0.5000, recall=0.6667.
- B: F1=0.7273, recall=0.7273.
- C: F1=0.5000, recall=0.5000.
- D: F1=0.6667, recall=0.5000.

Có 7 lỗi tại các sample `train_0042`, `train_0050`, `train_0051`, `train_0058`, `train_0060`, `train_0071`, `train_0089`. Parse rate 1.0 cho thấy prompt yêu cầu trả lời một chữ cái hoạt động ổn định; lỗi hiện tại là lỗi dự đoán, không phải lỗi parser.

### Nhận định

Zero-shot là baseline hợp lệ và tương đối mạnh trên subset nhỏ. Tuy nhiên accuracy 0.65 không thể ngoại suy cho toàn validation vì chỉ có 20 rows và phân bố lớp không cân bằng. Macro-F1 thấp hơn accuracy phản ánh hiệu năng không đồng đều giữa các lớp ít mẫu.

### Bước tiếp theo

Chạy `DEBUG_LIMIT=None` trên đủ 298 validation rows trước khi dùng baseline cho quyết định mô hình. Nên lưu confusion matrix và phân tích lỗi có nội dung câu hỏi/video; hiện `question_type=unknown` nên chưa thể xác định nhóm năng lực yếu.